# CS-808 Assignment 1: starter notebook (a scaffold, not a solution)

**What this is:** the workflow of the assignment as empty steps, plus a few helpers for boilerplate that is not part of the learning (making folders, computing a SHA-256, recording audit results, exporting them).

**What this is not:** it contains no API calls, no quality tests, no pass/fail thresholds, no plots and no option-specific analysis. **You make every substantive decision**: which source, which route, what "unfit" means for your intended use, which checks, which rules, which verdict.

**The workflow:** 1 configuration → 2 acquisition (once) → 3 save the raw response → 4 checksum and provenance → 5 load the raw data *offline* → 6 audit on six dimensions → 7 figures → 8 export the audit summary.

Read `../Assignment_01_Data_Acquisition_Audit.md` first. Your submitted code must run top to bottom, **offline**, from the files in `raw/`.

In [ ]:
import json, hashlib, platform
from datetime import datetime, timezone
from pathlib import Path

ROOT = Path.cwd()
RAW, OUTPUTS = ROOT / 'raw', ROOT / 'outputs'
DIMENSIONS = ['uniqueness', 'completeness', 'validity', 'consistency', 'timeliness', 'representativeness']


# ---- boilerplate helpers (not part of the assessment) -----------------------------------------------------------
def setup_dirs():
    RAW.mkdir(exist_ok=True); OUTPUTS.mkdir(exist_ok=True)

def utc_now():
    return datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as fh:
        for block in iter(lambda: fh.read(chunk), b''):
            h.update(block)
    return h.hexdigest()

def save_raw(content: bytes, name: str, ext: str):
    """Write bytes exactly as received to raw/<name>_<UTC>.<ext>. Refuses to overwrite: raw files are never edited."""
    path = RAW / f'{name}_{utc_now()}.{ext}'
    if path.exists():
        raise FileExistsError(path)
    path.write_bytes(content)
    return path

def list_raw():
    return sorted(p for p in RAW.iterdir() if p.is_file() and p.name != 'README.txt') if RAW.exists() else []

def new_audit_results():
    return {d: [] for d in DIMENSIONS}

def record_check(results, dimension, name, rule, observed, passed, note=''):
    """Store one check. YOU decide the rule and whether it passed; this function judges nothing."""
    if dimension not in results:
        raise ValueError(f'dimension must be one of {DIMENSIONS}')
    results[dimension].append({'dimension': dimension, 'check': name, 'rule_set_before_results': rule, 'observed': observed, 'passed': passed, 'note': note})

def audit_progress(results):
    for d in DIMENSIONS:
        print(f'{d:<20} {len(results[d])} check(s) recorded')
    print('total:', sum(len(v) for v in results.values()))

def export_audit(results):
    import csv
    OUTPUTS.mkdir(exist_ok=True)
    rows = [r for d in DIMENSIONS for r in results[d]]
    (OUTPUTS / 'audit_summary.json').write_text(json.dumps(rows, indent=2, default=str))
    with open(OUTPUTS / 'audit_summary.csv', 'w', newline='') as fh:
        w = csv.DictWriter(fh, fieldnames=['dimension', 'check', 'rule_set_before_results', 'observed', 'passed', 'note'])
        w.writeheader(); w.writerows(rows)
    print(f'wrote outputs/audit_summary.csv and .json ({len(rows)} checks)')

def save_figure(fig, name):
    OUTPUTS.mkdir(exist_ok=True)
    fig.savefig(OUTPUTS / f'{name}.png', dpi=200, bbox_inches='tight', facecolor='white')

def write_provenance(record: dict):
    """Write provenance.json; warns about fields still left as TODO or empty."""
    todo = [k for k, v in record.items() if v in ('', None, [], {}) or 'TODO' in json.dumps(v)]
    if todo:
        print('Provenance fields still to complete:', todo)
    (ROOT / 'provenance.json').write_text(json.dumps(record, indent=2))

setup_dirs()
print('folders ready:', RAW.name, OUTPUTS.name)

## 1. Configuration

Fill this in **first**, and write your rejection criteria and rules **before you look at any results**. Do not put keys, passwords or tokens here.

In [ ]:
STUDENT_ID = 'TODO'
OPTION = 'TODO'                      # 'A', 'B', 'C', 'D' or 'E'
SOURCE_NAME = 'TODO'
USER_AGENT = 'TODO: course name + your contact email (only where the service uses one)'

INTENDED_USE = 'TODO: one sentence: what question will this data help answer?'

REJECTION_CRITERIA = [               # what would make you reject this data? Write them BEFORE any results.
    # 'TODO',
]

RULES = {}                           # your pass/fail rules, decided BEFORE the audit: name -> description

def check_config():
    todo = [k for k, v in dict(STUDENT_ID=STUDENT_ID, OPTION=OPTION, SOURCE_NAME=SOURCE_NAME, INTENDED_USE=INTENDED_USE).items() if 'TODO' in v]
    if not REJECTION_CRITERIA: todo.append('REJECTION_CRITERIA')
    print('Still to fill in:', todo if todo else 'nothing')
check_config()

## 2. Acquisition (do this once; then keep the raw files)

Choose the **most direct permitted route** for your source and say why a more fragile method (HTML scraping, browser automation) was unnecessary. Rules: at most one request every 5 seconds per host; an honest `User-Agent` where the service uses one; back off on HTTP 429; no keys in this notebook; never store full copyrighted article text. Document the licence or terms (and `robots.txt` where relevant to web pages or feeds).

**Option B:** take one snapshot per day (at least 3 different days spanning at least 48 hours) and save each one separately.

In [ ]:
ACQUIRE = False        # set to True only on a day you collect; leave False so 'Run All' works offline
if ACQUIRE:
    # TODO: write your own acquisition code here.
    raise NotImplementedError('write your acquisition code here')
else:
    print('Acquisition skipped (ACQUIRE = False). Everything below runs from the files in raw/.')

## 3. Save the raw response

Save each response **exactly as received**, never edited, never overwritten. The helper `save_raw(content, name, ext)` writes `raw/<name>_<UTC>.<ext>` and refuses to overwrite. Put it inside your acquisition code above.

## 4. Checksum and provenance

Seven fields: source URL · query parameters · retrieval time (UTC) · who or what retrieved it · licence or terms (quoted) · SHA-256 of each raw file · known limitations. Fill the template; the file list and checksums below are only boilerplate. **Option B: one record per snapshot.**

In [ ]:
raw_files = list_raw()
print(len(raw_files), 'raw file(s) in raw/')
provenance = json.loads((ROOT / 'provenance_template.json').read_text()) if (ROOT / 'provenance_template.json').exists() else {}
provenance['files'] = [{'file': p.name, 'sha256': sha256_file(p), 'bytes': p.stat().st_size} for p in raw_files] or provenance.get('files', [])
# TODO: complete the remaining fields of `provenance` by hand (URL, parameters, time, licence wording, limitations).
# write_provenance(provenance)

## 5. Load the raw data (offline)

Read **only** from the files in `raw/`. There must be no network call from here to the end of the notebook.

In [ ]:
# TODO: load your raw file(s) from raw/ into a DataFrame (or several).
df = None

## 6. Audit on six dimensions

Write your rules in `RULES` **before** running any check. Aim for **at least 8 explicit checks covering all six dimensions**, each with a count. Questions to ask (these are prompts, not tests):

In [ ]:
RESULTS = new_audit_results()

### Uniqueness
What identifies one record? How do you know that key is unique? Duplicate *keys* or duplicate *records*? (Option B: duplicates within and across snapshots?)

In [ ]:
# TODO: your uniqueness checks. Record each one:
# record_check(RESULTS, 'uniqueness', name='...', rule='... (written before)', observed=..., passed=..., note='...')

### Completeness
What is the **full set of keys you expect** (dates, countries, items, outlets by days) and where does that expectation come from? Compare the data against it: **absent rows are invisible to a null count.** Where are values blank, and how does this source encode "missing"?

In [ ]:
# TODO: your completeness checks (missing values AND missing rows).

### Validity
Which values are impossible or meaningless for this variable? Are there **placeholders that look like real values**?

In [ ]:
# TODO: your validity checks.

### Consistency
Are units, scales, encodings, time zones, labels and **definitions** the same everywhere (across files, outlets, years)? Does the same label (a "year", a "country", a "price") mean the same thing in every place it appears?

In [ ]:
# TODO: your consistency checks.

### Timeliness
How old is the newest observation compared with when you retrieved it? What does the source say about when it was built, and whether published values are revised?

In [ ]:
# TODO: your timeliness checks.

### Representativeness
Who or what is **not** in the data, and which of your choices (source, filter, window, outlets) caused that?

In [ ]:
# TODO: your representativeness checks.

### Status is not quality
Which of your checks would have **passed even though the data was unfit** for your intended use (for example, a successful download)? Write them down.

In [ ]:
STATUS_IS_NOT_QUALITY = []     # TODO

### Your option's three extras
See the assignment sheet for the three extras for your option (A, B, C, D or E). Add cells here for them.

## 7. Figures

At least **two** Python figures with titles, units and labels: **Figure 1 exposes a quality problem**; Figure 2 is about the content. Save them with `save_figure(fig, 'figure_1')`.

In [ ]:
# TODO: Figure 1 (a quality problem)
# TODO: Figure 2 (the content)

## 8. Export the audit summary

In [ ]:
audit_progress(RESULTS)
export_audit(RESULTS)

## 9. Environment record (for the report's reproducibility note)

In [ ]:
import importlib
print('python', platform.python_version())
for pkg in ('pandas', 'numpy', 'matplotlib'):
    try:
        print(pkg, importlib.import_module(pkg).__version__)
    except Exception:
        print(pkg, 'not installed')

## Before you submit

Restart the kernel and **Run All** with the network off. Check: no keys in the notebook; raw files untouched; provenance complete; your rules were written before your results; the main report is at most 5 pages and the code is in Annex A.